# Training Dynamics

How do models learn? This notebook explores tools for analyzing training
dynamics — detecting phase transitions, grokking, circuit formation,
and changes in representation geometry over training.

This notebook covers the `irtk.training_dynamics` module.

## Why This Matters

Training dynamics analysis reveals when and how specific capabilities emerge during learning. Key phenomena like grokking (delayed generalization) and phase transitions (sudden capability jumps) provide clues about the relationship between memorization, generalization, and circuit formation.

**Key references:**
- [Olsson et al. (2022) "In-context Learning and Induction Heads"](https://arxiv.org/abs/2209.11895) — Discovers induction heads and training phase transitions

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt

import irtk
from irtk import training, training_dynamics

print("Ready!")

## 1. Train a Tiny Model

First, let's train a small model on modular addition — the classic
grokking task.

In [ ]:
# Create dataset
modulus = 7
data = training.create_algorithmic_dataset(
    "modular_addition", n_samples=modulus * modulus, modulus=modulus
)
print(f"Dataset: {data['tokens'].shape[0]} samples")
print(f"Example: {data['tokens'][0]} -> {data['labels'][0]}")

# Split into train/val
n = len(data["tokens"])
perm = np.random.RandomState(0).permutation(n)
train_idx = perm[:int(0.7 * n)]
val_idx = perm[int(0.7 * n):]

train_tokens = data["tokens"][train_idx]
train_labels = data["labels"][train_idx]
val_tokens = data["tokens"][val_idx]
val_labels = data["labels"][val_idx]
print(f"Train: {len(train_tokens)}, Val: {len(val_tokens)}")

In [ ]:
# Train with frequent checkpoints
cfg = irtk.HookedTransformerConfig(
    n_layers=1, d_model=32, n_ctx=4, d_head=16, n_heads=2,
    d_vocab=modulus + 1,  # tokens 0..modulus-1 + separator
)

result = training.train_tiny_model(
    cfg, train_tokens, train_labels,
    val_tokens=val_tokens, val_labels=val_labels,
    epochs=200, batch_size=32, lr=1e-3,
    checkpoint_every=10,
)
print(f"Final train loss: {result.train_losses[-1]:.4f}")
print(f"Final val acc: {result.val_accs[-1]:.4f}")
print(f"Checkpoints: {sorted(result.checkpoints.keys())}")

## 2. Detect Phase Transitions

Find sharp changes in the loss curve that indicate qualitative
shifts in what the model has learned.

In [ ]:
trans = training_dynamics.detect_phase_transitions(
    np.array(result.train_losses), window=5, threshold=2.0
)

print(f"Detected {len(trans['transition_indices'])} phase transitions")
print(f"At epochs: {trans['transition_indices']}")
print(f"Mean derivative: {trans['mean_derivative']:.6f}")

fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

# Training loss
axes[0].plot(result.train_losses, label="Train loss")
for idx in trans["transition_indices"]:
    axes[0].axvline(idx, color='red', alpha=0.5, linestyle='--')
axes[0].set_ylabel("Loss")
axes[0].set_title("Training Loss with Phase Transitions")
axes[0].legend()

# Smoothed derivative
axes[1].plot(trans["smoothed_derivative"], color='orange')
axes[1].axhline(trans["mean_derivative"] * 2.0, color='red', linestyle='--',
                label=f'Threshold ({trans["mean_derivative"] * 2.0:.4f})')
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("|d(loss)/d(epoch)|")
axes[1].set_title("Smoothed Rate of Change")
axes[1].legend()

plt.tight_layout()
plt.show()

## 3. Grokking Analysis

Grokking is when a model memorizes the training set quickly but
takes much longer to generalize. Let's check for it.

In [ ]:
grok = training_dynamics.grokking_analysis(
    result.train_losses, result.val_accs,
    memorization_threshold=0.95,
    generalization_threshold=0.90,
)

print(f"Grokking detected: {grok['has_grokking']}")
print(f"Memorization epoch: {grok['memorization_epoch']}")
print(f"Generalization epoch: {grok['generalization_epoch']}")
print(f"Grokking gap: {grok['grokking_gap']} epochs")

fig, ax1 = plt.subplots(figsize=(12, 5))
ax1.plot(result.train_losses, 'b-', label='Train Loss')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss', color='b')

ax2 = ax1.twinx()
ax2.plot(result.val_accs, 'r-', label='Val Accuracy')
ax2.set_ylabel('Accuracy', color='r')

if grok['memorization_epoch'] is not None:
    ax1.axvline(grok['memorization_epoch'], color='blue', linestyle='--',
                alpha=0.5, label=f'Memorization (epoch {grok["memorization_epoch"]})')
if grok['generalization_epoch'] is not None:
    ax1.axvline(grok['generalization_epoch'], color='red', linestyle='--',
                alpha=0.5, label=f'Generalization (epoch {grok["generalization_epoch"]})')

ax1.legend(loc='upper left')
ax2.legend(loc='upper right')
ax1.set_title('Grokking Analysis')
plt.tight_layout()
plt.show()

## 4. Circuit Formation Trajectory

Track how a circuit-level metric evolves across training checkpoints.

In [ ]:
# Create test sequences
test_seqs = [jnp.array(data["tokens"][i]) for i in range(10)]

# Track attention entropy as a circuit metric
def entropy_metric(model, tokens):
    _, cache = model.run_with_cache(tokens)
    pattern = cache.cache_dict.get("blocks.0.attn.hook_pattern")
    if pattern is None:
        return 0.0
    # Average entropy across heads and positions
    p = np.array(pattern)
    p = np.clip(p, 1e-10, 1.0)
    ent = -np.sum(p * np.log(p), axis=-1)
    return float(np.mean(ent))

traj = training_dynamics.circuit_formation_trajectory(
    result.checkpoints, test_seqs, entropy_metric
)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(traj["epochs"], traj["metrics"], 'o-')
ax.set_xlabel("Training Epoch")
ax.set_ylabel("Attention Entropy")
ax.set_title("Attention Entropy During Training")
plt.tight_layout()
plt.show()

## 5. Loss Landscape Slice

Visualize how the loss changes along a random direction in parameter space.

In [ ]:
landscape = training_dynamics.loss_landscape_slice(
    result.model, train_tokens, train_labels,
    alphas=np.linspace(-2, 2, 41),
)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(landscape["alphas"], landscape["losses"], 'o-')
ax.axvline(0, color='red', linestyle='--', alpha=0.5, label='Trained model')
ax.set_xlabel("Perturbation magnitude (α)")
ax.set_ylabel("Loss")
ax.set_title(f"Loss Landscape Slice (direction norm={landscape['direction_norm']:.2f})")
ax.legend()
plt.tight_layout()
plt.show()

## 6. Weight Norm Trajectory

Track how weight magnitudes evolve during training — sharp increases
or decreases often coincide with phase transitions.

In [ ]:
norms = training_dynamics.weight_norm_trajectory(result.checkpoints)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Total norm
axes[0].plot(norms["epochs"], norms["total_norm"], 'o-', linewidth=2)
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Total Parameter Norm")
axes[0].set_title("Total Weight Norm")

# Per-component
for name in ["W_E", "W_U", "L0_W_Q", "L0_W_K", "L0_W_V", "L0_W_O"]:
    if name in norms["per_component"]:
        axes[1].plot(norms["epochs"], norms["per_component"][name], 'o-', label=name)
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Frobenius Norm")
axes[1].set_title("Per-Component Norms")
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.show()

## 7. Effective Rank Trajectory

Track how the effective dimensionality of representations changes
during training.

In [ ]:
rank_traj = training_dynamics.effective_rank_trajectory(
    result.checkpoints, test_seqs, "blocks.0.hook_resid_post"
)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(rank_traj["epochs"], rank_traj["effective_rank"], 'o-')
ax.set_xlabel("Training Epoch")
ax.set_ylabel("Effective Rank (Participation Ratio)")
ax.set_title("Representation Dimensionality During Training")
plt.tight_layout()
plt.show()

## Summary

| Function | Purpose |
|----------|--------|
| `detect_phase_transitions()` | Find sharp changes in a metric time series |
| `grokking_analysis()` | Detect grokking (memorization → generalization gap) |
| `circuit_formation_trajectory()` | Track a circuit metric across training checkpoints |
| `loss_landscape_slice()` | 1D slice of loss landscape along a direction |
| `weight_norm_trajectory()` | Track weight norms over training |
| `effective_rank_trajectory()` | Track effective dimensionality of activations |